# Notebook 6: Bayesian A/B Testing

## Overview
This notebook introduces **Bayesian thinking** for A/B testing—a fundamentally different philosophical approach from the frequentist methods in Notebooks 2-5.

### Learning Objectives
- Understand Bayesian inference: prior + data → posterior
- Learn the Beta-Binomial model for binary outcomes
- Calculate probability of being best variant
- Decide using expected loss framework
- Compare Bayesian vs Frequentist approaches

### Why Bayesian?
1. **Intuitive interpretation**: "What's the probability this variant is better?" (exactly what practitioners want)
2. **Incorporates prior knowledge**: Can use historical data or domain expertise
3. **Sequential-friendly**: Easy to update beliefs as data arrives
4. **Decision-focused**: Built for making business decisions, not just statistical testing

### A Simple Analogy
- **Frequentist**: "If the coin were truly fair, how likely is this many heads?" (probability of data given hypothesis)
- **Bayesian**: "Given I saw this many heads, what's the probability the coin is fair?" (probability of hypothesis given data)

The Bayesian answer is usually what people actually want.

In [1]:
import os
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams['figure.figsize'] = (12, 6)

# Create output directory
os.makedirs('../data/outputs/nb06', exist_ok=True)


In [2]:
# Load cleaned data
data_path = Path("../data/outputs/nb01/nb01_hillstrom_clean.csv")
df = pd.read_csv(data_path)

print(f"Data shape: {df.shape}")
print(f"\nGroup stats:\n{df.groupby('segment')['conversion'].agg(['sum', 'count', 'mean'])}")

Data shape: (64000, 24)

Group stats:
               sum  count      mean
segment                            
Mens E-Mail    267  21307  0.012531
No E-Mail      122  21306  0.005726
Womens E-Mail  189  21387  0.008837


## Concept: Bayesian vs Frequentist Thinking

### Side-by-Side Comparison

| Aspect | Frequentist | Bayesian |
|--------|-------------|----------|
| **Probability** | Long-run frequency | Degree of belief |
| **Parameter** | Fixed unknown constant | Random variable with distribution |
| **Prior** | Irrelevant (or wrong to use) | Core of inference |
| **Inference** | P(data \| H₀) — probability of data under null | P(H \| data) — probability of hypothesis given data |
| **Conclusion** | "Reject H₀" or "Fail to reject" | "Parameter is likely in range [a, b]" |
| **Question** | "How often would this occur if H₀ true?" | "What do I believe about the parameter?" |
| **Early stopping** | Inflates Type I error | No problem! Just update beliefs |
| **CI interpretation** | Long-run coverage property | Direct probability interval |

### Example: Coin Flip
You flip a coin 10 times, get 8 heads. Is it fair?

**Frequentist**: P(8+ heads \| fair coin) ≈ 0.055. At α=0.05, just barely fail to reject fairness.

**Bayesian**: P(coin is fair \| 8 heads) is low. Prior belief matters:
- If you believed it was fair: "I'm now skeptical"
- If you knew it might be biased: "This confirms it's biased"

### Why Bayesian is Natural for A/B Testing
In business, we don't care about the probability of data under H₀. We care about:
- "What's P(variant A is better than variant B)?"
- "What's the expected loss if I pick variant A?"
- "Can I stop early and declare a winner?"

All of these are natural Bayesian questions.

## The Beta Distribution as a Prior

### What is Beta?
The **Beta distribution** is a flexible probability distribution on [0, 1], perfect for modeling uncertainty about a probability.

Parameters:
- **α** (alpha): shape parameter, roughly "successes + 1"
- **β** (beta): shape parameter, roughly "failures + 1"
- **Mean** = α / (α + β)
- **Variance** decreases with larger α + β (more concentrated)

### Intuitive Interpretation
Think of Beta(α, β) as encoding: "I've seen α successes and β failures."

- **Beta(1, 1)**: Uniform — "completely uninformed"
- **Beta(10, 10)**: Concentrated at 0.5 — "I'm pretty sure it's around 50%"
- **Beta(100, 100)**: Tightly concentrated at 0.5 — "I'm very confident it's ~50%"
- **Beta(90, 10)**: Peaked at 0.9 — "I believe it's around 90%, pretty confident"

### Why Conjugate?
Beta is the **conjugate prior** for binomial data. This means:
- **Prior**: Beta(α, β)
- **Data**: X successes out of n trials
- **Posterior**: Beta(α + X, β + n - X)

**Conjugacy is powerful**: the posterior has the same form as the prior! No complex math needed.

In [3]:
# (Chart cell removed — interactive Plotly version rendered below in the "Blog-Ready Plotly Charts" section.)

## Beta-Binomial Model for Conversion

### The Model
For each variant, we model:
1. **Prior**: Beta(α₀, β₀) — what we believe before seeing data
2. **Likelihood**: Binomial(p | X successes, n trials)
3. **Posterior**: Beta(α₀ + X, β₀ + n - X) — updated belief

### Interpretation
After observing data:
- **Posterior α** = prior α + observed successes
- **Posterior β** = prior β + observed failures

The posterior is centered at:
(α₀ + X) / (α₀ + β₀ + n)

With weak prior (α₀ = β₀ = 1), this is approximately the sample conversion rate.
With strong prior (α₀ = β₀ = 100), the posterior is pulled toward 0.5 (the prior belief).

### Advantages
- **No distributional assumptions**: Works exactly for binary outcomes
- **Closed-form posterior**: Easy calculation
- **Natural uncertainty**: The posterior spread captures sampling uncertainty
- **Easy updates**: New data → new posterior (which becomes next prior)

In [4]:
# Calculate posteriors for each group using Beta-Binomial model
# Prior: weakly informative Beta(1, 1)

groups = df['segment'].unique()
posterior_params = {}

print("=== Beta-Binomial Posterior Analysis (Conversion) ===\n")

for group in groups:
    group_data = df[df['segment'] == group]
    conversions = group_data['conversion'].sum()
    failures = len(group_data) - conversions
    
    # Prior
    alpha_prior = 1
    beta_prior = 1
    
    # Posterior
    alpha_post = alpha_prior + conversions
    beta_post = beta_prior + failures
    
    posterior_params[group] = (alpha_post, beta_post)
    
    # Posterior mean and credible interval
    post_mean = alpha_post / (alpha_post + beta_post)
    post_var = (alpha_post * beta_post) / ((alpha_post + beta_post)**2 * (alpha_post + beta_post + 1))
    post_std = np.sqrt(post_var)
    
    # 95% HDI (High Density Interval) using quantiles
    post_dist = stats.beta(alpha_post, beta_post)
    hdi_lower = post_dist.ppf(0.025)
    hdi_upper = post_dist.ppf(0.975)
    
    print(f"{group}:")
    print(f"  Data: {conversions} conversions / {len(group_data)} customers")
    print(f"  Posterior: Beta({alpha_post}, {beta_post})")
    print(f"  Posterior mean: {post_mean:.4f}")
    print(f"  Posterior std: {post_std:.4f}")
    print(f"  95% HDI: [{hdi_lower:.4f}, {hdi_upper:.4f}]")
    print()

# Store for later use
posterior_df = pd.DataFrame([
    {
        'Segment': group,
        'Alpha': posterior_params[group][0],
        'Beta': posterior_params[group][1],
        'Mean': posterior_params[group][0] / (posterior_params[group][0] + posterior_params[group][1]),
        'HDI_Lower': stats.beta(posterior_params[group][0], posterior_params[group][1]).ppf(0.025),
        'HDI_Upper': stats.beta(posterior_params[group][0], posterior_params[group][1]).ppf(0.975)
    }
    for group in groups
])

print(posterior_df.to_string(index=False))

=== Beta-Binomial Posterior Analysis (Conversion) ===

Womens E-Mail:
  Data: 189 conversions / 21387 customers
  Posterior: Beta(190, 21199)
  Posterior mean: 0.0089
  Posterior std: 0.0006
  95% HDI: [0.0077, 0.0102]

No E-Mail:
  Data: 122 conversions / 21306 customers
  Posterior: Beta(123, 21185)
  Posterior mean: 0.0058
  Posterior std: 0.0005
  95% HDI: [0.0048, 0.0068]

Mens E-Mail:
  Data: 267 conversions / 21307 customers
  Posterior: Beta(268, 21041)
  Posterior mean: 0.0126
  Posterior std: 0.0008
  95% HDI: [0.0111, 0.0141]

      Segment  Alpha  Beta     Mean  HDI_Lower  HDI_Upper
Womens E-Mail    190 21199 0.008883   0.007670   0.010183
    No E-Mail    123 21185 0.005772   0.004800   0.006833
  Mens E-Mail    268 21041 0.012577   0.011124   0.014116


In [5]:
# (Chart cell removed — interactive Plotly version rendered below in the "Blog-Ready Plotly Charts" section.)

## Probability of Being Best

### The Key Insight
Instead of hypothesis testing (yes/no), Bayesian A/B testing gives us probabilities:
- P(Mens Email is best)
- P(Womens Email is best)
- P(No Email is best)

These sum to 100% and directly answer business questions.

### Method
1. Draw samples from each variant's posterior
2. For each sample set, determine which is highest
3. Count the frequency

This is **Monte Carlo estimation** and is fast even for many variants.

In [6]:
# Monte Carlo: Draw samples from posteriors and calculate probabilities
np.random.seed(42)

n_samples = 100000

# Draw samples from each posterior
samples = {}
for group in groups:
    alpha, beta = posterior_params[group]
    samples[group] = np.random.beta(alpha, beta, n_samples)

# Calculate probabilities
prob_best = {}

for group in groups:
    # Count how many times this group has the highest conversion rate
    is_best = samples[group] > samples[[g for g in groups if g != group][0]]
    
    for other_group in groups:
        if other_group != group:
            is_best = is_best & (samples[group] > samples[other_group])
    
    prob_best[group] = np.mean(is_best)

print("\n=== Probability of Being Best (Monte Carlo) ===\n")
for group in groups:
    print(f"{group}: {prob_best[group]:.4f} ({prob_best[group]*100:.2f}%)")

# Pairwise comparisons
print("\n=== Pairwise Comparisons ===\n")

for i, group1 in enumerate(groups):
    for group2 in groups[i+1:]:
        prob_g1_better = np.mean(samples[group1] > samples[group2])
        prob_g2_better = np.mean(samples[group2] > samples[group1])
        
        print(f"P({group1} > {group2}): {prob_g1_better:.4f}")
        print(f"P({group2} > {group1}): {prob_g2_better:.4f}")
        print()


=== Probability of Being Best (Monte Carlo) ===

Womens E-Mail: 0.0001 (0.01%)
No E-Mail: 0.0000 (0.00%)
Mens E-Mail: 0.9999 (99.99%)

=== Pairwise Comparisons ===

P(Womens E-Mail > No E-Mail): 0.9999
P(No E-Mail > Womens E-Mail): 0.0001

P(Womens E-Mail > Mens E-Mail): 0.0001
P(Mens E-Mail > Womens E-Mail): 0.9999

P(No E-Mail > Mens E-Mail): 0.0000
P(Mens E-Mail > No E-Mail): 1.0000



In [7]:
# (Chart cell removed — interactive Plotly version rendered below in the "Blog-Ready Plotly Charts" section.)

## Expected Loss: A Decision Framework

### The Idea
For each variant, we can compute the **expected loss** of choosing it:

**E[Loss(choosing A)] = ∫ Loss(A vs true param) × P(true param) d(param)**

A simple loss function: Loss = max(0, best_variant - chosen_variant)

If the best variant has conversion rate 0.05 and we choose a variant with 0.04, the loss is 0.01.

### Why This Matters
- Accounts for probability of being best AND the magnitude of potential loss
- Handles multiple variants naturally
- Focuses on what matters: minimizing regret, not just hypothesis testing

In [8]:
# Calculate expected loss for each variant
def expected_loss(chosen_variant_samples, other_samples_list):
    """
    Calculate expected loss if we choose a given variant.
    Loss = max(0, max(others) - chosen)
    """
    max_other = np.max(np.array(other_samples_list), axis=0)
    loss = np.maximum(0, max_other - chosen_variant_samples)
    return np.mean(loss)

print("=== Expected Loss Analysis ===\n")

expected_losses = {}

for group in groups:
    other_samples = [samples[g] for g in groups if g != group]
    exp_loss = expected_loss(samples[group], other_samples)
    expected_losses[group] = exp_loss
    
    print(f"{group}: {exp_loss:.6f}")

# Identify best variant by expected loss
best_variant = min(expected_losses, key=expected_losses.get)
print(f"\nBest choice by expected loss: {best_variant}")

# Visualize expected loss

=== Expected Loss Analysis ===

Womens E-Mail: 0.003687
No E-Mail: 0.006802
Mens E-Mail: 0.000000

Best choice by expected loss: Mens E-Mail


## Credible Intervals vs Confidence Intervals

### Key Difference
- **Confidence Interval (Frequentist)**: "If we repeated this experiment many times, 95% of our intervals would contain the true parameter"
  - The parameter is fixed, the interval is random
  - Probability is about the *procedure*, not this specific interval

- **Credible Interval (Bayesian)**: "Given the data, there's a 95% probability the parameter is in this interval"
  - The parameter is random (has a distribution), the interval is fixed
  - Probability is directly about the parameter value

### Highest Density Interval (HDI)
The Bayesian credible interval that includes the highest posterior density. For our Beta posteriors, we can compute this directly.

In [9]:
# Compare CIs and credible intervals
ci_comparison = []

for group in groups:
    alpha_post, beta_post = posterior_params[group]
    posterior = stats.beta(alpha_post, beta_post)
    
    # Bayesian HDI (95%)
    hdi_lower = posterior.ppf(0.025)
    hdi_upper = posterior.ppf(0.975)
    
    # For comparison, analytical Wald CI (from earlier notebooks)
    group_data = df[df['segment'] == group]
    conversions = group_data['conversion'].sum()
    n = len(group_data)
    p = conversions / n
    se = np.sqrt(p * (1 - p) / n)
    z = stats.norm.ppf(0.975)
    
    wald_lower = p - z * se
    wald_upper = p + z * se
    
    ci_comparison.append({
        'Segment': group,
        'Bayesian HDI Lower': f"{hdi_lower:.4f}",
        'Bayesian HDI Upper': f"{hdi_upper:.4f}",
        'Wald CI Lower': f"{wald_lower:.4f}",
        'Wald CI Upper': f"{wald_upper:.4f}",
        'HDI Width': f"{hdi_upper - hdi_lower:.4f}",
        'Wald Width': f"{wald_upper - wald_lower:.4f}"
    })

ci_df = pd.DataFrame(ci_comparison)
print("\n=== Bayesian Credible Interval vs Frequentist CI ===\n")
print(ci_df.to_string(index=False))

# Save comparison
ci_df.to_csv('../data/outputs/nb06/nb06_ci_bayesian_comparison.csv', index=False)


=== Bayesian Credible Interval vs Frequentist CI ===

      Segment Bayesian HDI Lower Bayesian HDI Upper Wald CI Lower Wald CI Upper HDI Width Wald Width
Womens E-Mail             0.0077             0.0102        0.0076        0.0101    0.0025     0.0025
    No E-Mail             0.0048             0.0068        0.0047        0.0067    0.0020     0.0020
  Mens E-Mail             0.0111             0.0141        0.0110        0.0140    0.0030     0.0030


## Sensitivity Analysis: Prior Choice

One criticism of Bayesian methods: "Results depend on the prior!"

This is partly true, but with large samples, the prior matters less. Let's test this.

In [10]:
# Test multiple priors
priors_to_test = [
    (1, 1, "Uninformed Beta(1,1)"),
    (5, 5, "Weak Beta(5,5)"),
    (10, 10, "Moderate Beta(10,10)"),
    (20, 20, "Strong Beta(20,20)")
]

sensitivity_results = []

for alpha_prior, beta_prior, prior_name in priors_to_test:
    print(f"\nPrior: {prior_name}")
    print("-" * 50)
    
    prior_probs_best = {}
    
    for group in groups:
        group_data = df[df['segment'] == group]
        conversions = group_data['conversion'].sum()
        failures = len(group_data) - conversions
        
        # Posterior with this prior
        alpha_post = alpha_prior + conversions
        beta_post = beta_prior + failures
        
        # Sample
        samp = np.random.beta(alpha_post, beta_post, 50000)
        prior_probs_best[group] = samp
    
    # Calculate prob best
    prob_best_prior = {}
    for group in groups:
        is_best = prior_probs_best[group] > prior_probs_best[[g for g in groups if g != group][0]]
        for other in groups:
            if other != group:
                is_best = is_best & (prior_probs_best[group] > prior_probs_best[other])
        prob_best_prior[group] = np.mean(is_best)
    
    for group in groups:
        sensitivity_results.append({
            'Prior': prior_name,
            'Segment': group,
            'P(Best)': f"{prob_best_prior[group]:.4f}"
        })
    
    print(f"P(Men's Best): {prob_best_prior[groups[0]]:.4f}")
    if len(groups) > 1:
        print(f"P(Women's Best): {prob_best_prior[groups[1]]:.4f}")
    if len(groups) > 2:
        print(f"P(Control Best): {prob_best_prior[groups[2]]:.4f}")

sensitivity_df = pd.DataFrame(sensitivity_results)

print("\n\n=== SENSITIVITY ANALYSIS SUMMARY ===\n")
# Pivot for readability
pivot_sensitivity = sensitivity_df.pivot(index='Segment', columns='Prior', values='P(Best)')
print(pivot_sensitivity)

print("\n\nConclusion: With large samples (64K customers), prior choice has minimal impact.")
print("P(Best) is similar across all reasonable priors.")


Prior: Uninformed Beta(1,1)
--------------------------------------------------
P(Men's Best): 0.0001
P(Women's Best): 0.0000
P(Control Best): 0.9999

Prior: Weak Beta(5,5)
--------------------------------------------------
P(Men's Best): 0.0001
P(Women's Best): 0.0000
P(Control Best): 0.9999

Prior: Moderate Beta(10,10)
--------------------------------------------------
P(Men's Best): 0.0001
P(Women's Best): 0.0000
P(Control Best): 0.9999

Prior: Strong Beta(20,20)
--------------------------------------------------
P(Men's Best): 0.0002
P(Women's Best): 0.0000
P(Control Best): 0.9998


=== SENSITIVITY ANALYSIS SUMMARY ===

Prior         Moderate Beta(10,10) Strong Beta(20,20) Uninformed Beta(1,1)  \
Segment                                                                      
Mens E-Mail                 0.9999             0.9998               0.9999   
No E-Mail                   0.0000             0.0000               0.0000   
Womens E-Mail               0.0001             0.0002   

## Extension: Bayesian Analysis for Spend (Normal-Normal Model)

For continuous outcomes like spend, we use a **Normal-Normal model**:
- **Prior**: N(μ₀, σ₀²)
- **Likelihood**: N(ȳ, σ²/n)
- **Posterior**: N(μₙ, σₙ²) with:
  - μₙ = (σ₀⁻² μ₀ + σ⁻² ȳ n) / (σ₀⁻² + σ⁻² n)
  - σₙ² = 1 / (σ₀⁻² + σ⁻² n)

The posterior mean is a weighted average of prior mean and observed sample mean.

In [11]:
# Bayesian analysis for spend
# Prior: N(mean_spend, sd_spend) based on control group
control_data = df[df['segment'] == "No E-Mail"]
prior_mean = control_data['spend'].mean()
prior_sd = control_data['spend'].std()

print(f"\n=== Bayesian Analysis for Spend ===\n")
print(f"Prior (from control): N({prior_mean:.2f}, {prior_sd:.2f}²)")
print()

spend_results = []

for group in groups:
    group_data = df[df['segment'] == group]
    y_bar = group_data['spend'].mean()
    y_sd = group_data['spend'].std()
    n = len(group_data)
    
    # Posterior parameters
    prior_prec = 1 / (prior_sd ** 2)
    likelihood_prec = n / (y_sd ** 2)
    
    posterior_prec = prior_prec + likelihood_prec
    posterior_mean = (prior_prec * prior_mean + likelihood_prec * y_bar) / posterior_prec
    posterior_sd = 1 / np.sqrt(posterior_prec)
    
    # 95% credible interval
    z = stats.norm.ppf(0.975)
    ci_lower = posterior_mean - z * posterior_sd
    ci_upper = posterior_mean + z * posterior_sd
    
    print(f"{group}:")
    print(f"  Sample mean: ${y_bar:.2f}")
    print(f"  Posterior: N({posterior_mean:.2f}, {posterior_sd:.2f}²)")
    print(f"  95% Credible Interval: [${ci_lower:.2f}, ${ci_upper:.2f}]")
    print()
    
    spend_results.append({
        'Segment': group,
        'Sample Mean': y_bar,
        'Posterior Mean': posterior_mean,
        'Posterior SD': posterior_sd,
        'CI Lower': ci_lower,
        'CI Upper': ci_upper
    })

spend_df = pd.DataFrame(spend_results)


=== Bayesian Analysis for Spend ===

Prior (from control): N(0.65, 11.59²)

Womens E-Mail:
  Sample mean: $1.08
  Posterior: N(1.08, 0.10²)
  95% Credible Interval: [$0.87, $1.28]

No E-Mail:
  Sample mean: $0.65
  Posterior: N(0.65, 0.08²)
  95% Credible Interval: [$0.50, $0.81]

Mens E-Mail:
  Sample mean: $1.42
  Posterior: N(1.42, 0.12²)
  95% Credible Interval: [$1.18, $1.66]



In [12]:
# Create comprehensive Bayesian results summary
print("\n\n=== BAYESIAN A/B TEST SUMMARY ===\n")

summary_bayesian = pd.DataFrame([
    {
        'Segment': group,
        'N': len(df[df['segment'] == group]),
        'Conv Rate': f"{df[df['segment'] == group]['conversion'].mean():.4f}",
        'P(Best)': f"{prob_best[group]:.4f}",
        'Expected Loss': f"{expected_losses[group]:.6f}",
        'Recommended': 'YES' if group == best_variant else ''
    }
    for group in groups
])

print(summary_bayesian.to_string(index=False))

# Save results
summary_bayesian.to_csv('../data/outputs/nb06/nb06_bayesian_results.csv', index=False)

print("\n\n=== KEY INSIGHTS ===\n")
print(f"1. {best_variant} has lowest expected loss → Recommended choice")
print(f"\n2. Probability interpretation:")
for group in groups:
    print(f"   {group}: {prob_best[group]*100:.1f}% chance it's the best")

print(f"\n3. Full uncertainty captured by posterior distributions")
print(f"\n4. Sequential analysis-friendly: update as data arrives")
print(f"\n5. Results published to: ../data/outputs/nb06/nb06_bayesian_results.csv")



=== BAYESIAN A/B TEST SUMMARY ===

      Segment     N Conv Rate P(Best) Expected Loss Recommended
Womens E-Mail 21387    0.0088  0.0001      0.003687            
    No E-Mail 21306    0.0057  0.0000      0.006802            
  Mens E-Mail 21307    0.0125  0.9999      0.000000         YES


=== KEY INSIGHTS ===

1. Mens E-Mail has lowest expected loss → Recommended choice

2. Probability interpretation:
   Womens E-Mail: 0.0% chance it's the best
   No E-Mail: 0.0% chance it's the best
   Mens E-Mail: 100.0% chance it's the best

3. Full uncertainty captured by posterior distributions

4. Sequential analysis-friendly: update as data arrives

5. Results published to: ../data/outputs/nb06/nb06_bayesian_results.csv


## Summary: Bayesian vs Frequentist for A/B Testing

### Frequentist Approach (Notebooks 2-5)
**Strengths**:
- Objective (no prior choice)
- Well-established, understood by regulators
- Controls Type I error rigorously

**Weaknesses**:
- Answers the "wrong" question (probability of data vs probability of hypothesis)
- Hypothesis testing is binary (reject/fail to reject) — not useful for business
- Early stopping inflates error rates
- Confidence intervals don't directly mean what people think

### Bayesian Approach (This Notebook)
**Strengths**:
- Direct answer to business questions: "Which variant is best?"
- Intuitive interpretation of intervals
- Sequential analysis is natural and unbiased
- Incorporates prior knowledge
- Provides probabilities of each outcome

**Weaknesses**:
- Requires choosing a prior (though impact diminishes with data)
- Computationally more involved (though usually fast)
- Less familiar to some practitioners

### When to Use Each
- **Bayesian**: Business decisions, online experiments, when you want interpretable probabilities
- **Frequentist**: Regulatory requirements, pre-registered designs, when prior information is unavailable
- **Best Practice**: Use both! Agreement across methods increases confidence.

---

## Blog-Ready Plotly Charts

The cells below regenerate the charts from this notebook as responsive Plotly
HTML files for embedding in the blog post. They are **self-contained**: each
one re-loads the clean dataset from nb01 and re-derives the statistics it
needs, so you can run this section in isolation.

Outputs are written to `data/outputs/nb##/` with the suffix `_interactive.html`.

**Required packages:** `plotly` (install with `pip install plotly` if missing).

In [13]:
# ============================================================
# Blog-Ready Plotly Charts — self-contained, embed-friendly
# ============================================================
# These cells produce responsive Plotly HTML files for the blog post.
# They re-load from the nb01 clean CSV and re-derive stats so the section
# runs standalone. Each figure uses:
#   - include_plotlyjs='cdn' (single shared CDN load on the blog page)
#   - config={'responsive': True} so it resizes to container width
#   - automargin=True on axes + generous margins so labels never clip
#   - rotated tick labels on long categories, headroom for outside labels
import os, numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "notebook_connected"  # inline-render Plotly in cell output

OUT_DIR = os.path.abspath("../data/outputs/nb06")
os.makedirs(OUT_DIR, exist_ok=True)
CLEAN_CSV = os.path.abspath("../data/outputs/nb01/nb01_hillstrom_clean.csv")
df_blog = pd.read_csv(CLEAN_CSV)

# Shared palette aligned with nb01 Plotly charts
COLORS = {
    "Mens E-Mail": "#4C8BB8", "Womens E-Mail": "#5FA85F", "No E-Mail": "#E89B4C",
    "Match": "#2ECC71", "Mismatch": "#E74C3C", "Mixed": "#F39C12", "Control": "#95A5A6",
    "Treatment (Any Email)": "#4C8BB8",
}
PLOTLY_KW = dict(include_plotlyjs="cdn", full_html=True,
                 config={"responsive": True, "displaylogo": False})
BASE_LAYOUT = dict(template="plotly_white",
                   font=dict(family="Arial, sans-serif", size=13),
                   title_x=0.5,
                   margin=dict(l=70, r=40, t=90, b=90),
                   hoverlabel=dict(bgcolor="white", font_size=12))
print(f"Blog-ready Plotly charts will be written to: {OUT_DIR}")


Blog-ready Plotly charts will be written to: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/ab_testing/data/outputs/nb06


In [16]:
# Chart 1: Prior vs Posterior overlay per arm (conversion)
from scipy.stats import beta as _beta
prior_a, prior_b = 1.0, 1.0  # Beta(1,1) = Uniform
segs = ["Mens E-Mail", "Womens E-Mail", "No E-Mail"]
x = np.linspace(0, 0.03, 500)
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=_beta.pdf(x, prior_a, prior_b),
                         mode="lines", line=dict(color="#95A5A6", dash="dash", width=2),
                         name="Prior Beta(1,1)"))
for seg in segs:
    g = df_blog[df_blog["segment"]==seg]["conversion"]
    a = prior_a + g.sum(); b = prior_b + (len(g) - g.sum())
    fig.add_trace(go.Scatter(x=x, y=_beta.pdf(x, a, b), mode="lines",
                             line=dict(color=COLORS[seg], width=3), name=f"{seg} posterior",
                             hovertemplate=f"<b>{seg}</b><br>p: %{{x:.4f}}<br>density: %{{y:.1f}}<extra></extra>"))
fig.update_layout(**BASE_LAYOUT,
                  title="Prior vs Posterior Conversion Distributions per Arm",
                  xaxis=dict(title="Conversion Probability", automargin=True, tickformat=".2%"),
                  yaxis=dict(title="Density", automargin=True), height=520,
                  legend=dict(orientation="h", y=-0.2, x=0.5, xanchor="center"))
fig.write_html(os.path.join(OUT_DIR, "nb06_prior_posterior_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb06_prior_posterior_interactive.html")

# Chart 2: Posterior distribution of the difference (Mens - Control)
np.random.seed(42); N=20000
m = df_blog[df_blog["segment"]=="Mens E-Mail"]["conversion"]
c = df_blog[df_blog["segment"]=="No E-Mail"]["conversion"]
ps_m = _beta.rvs(prior_a + m.sum(), prior_b + len(m) - m.sum(), size=N)
ps_c = _beta.rvs(prior_a + c.sum(), prior_b + len(c) - c.sum(), size=N)
diff = (ps_m - ps_c) * 100
lo, hi = np.percentile(diff, [2.5, 97.5])
prob_pos = float((diff > 0).mean())
fig = go.Figure(go.Histogram(x=diff, nbinsx=60,
                             marker=dict(color="#4C8BB8", line=dict(color="black", width=0.5)),
                             hovertemplate="Diff: %{x:.3f} pp<br>Count: %{y}<extra></extra>"))
fig.add_vline(x=0, line_color="black")
fig.add_vline(x=lo, line_dash="dash", line_color="#E74C3C",
              annotation_text=f"2.5% = {lo:.3f}", annotation_position="top left")
fig.add_vline(x=hi, line_dash="dash", line_color="#E74C3C",
              annotation_text=f"97.5% = {hi:.3f}", annotation_position="top right")
fig.update_layout(**BASE_LAYOUT,
                  title=f"Posterior of Mens − Control Conversion Difference<br>"
                        f"<sub>P(Mens > Control) = {prob_pos:.3%} · 95% CrI [{lo:.3f}, {hi:.3f}] pp</sub>",
                  xaxis=dict(title="Difference (pp)", automargin=True),
                  yaxis=dict(title="Count", automargin=True), height=500, bargap=0.02)
fig.write_html(os.path.join(OUT_DIR, "nb06_posterior_difference_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb06_posterior_difference_interactive.html")

# Chart 3: Probability (Mens > Control) evolving over sample size
steps = np.linspace(200, min(len(m), len(c)), 30).astype(int)
probs=[]
for n in steps:
    m_n = m.iloc[:n]; c_n = c.iloc[:n]
    sm = _beta.rvs(prior_a + m_n.sum(), prior_b + len(m_n)-m_n.sum(), size=5000)
    sc = _beta.rvs(prior_a + c_n.sum(), prior_b + len(c_n)-c_n.sum(), size=5000)
    probs.append((sm > sc).mean())
fig = go.Figure(go.Scatter(x=steps, y=probs, mode="lines+markers",
                           line=dict(color="#4C8BB8", width=3),
                           marker=dict(size=7),
                           hovertemplate="n: %{x:,}<br>P(M>C): %{y:.3%}<extra></extra>"))
fig.add_hline(y=0.95, line_dash="dash", line_color="#2ECC71",
              annotation_text="95% decision threshold")
fig.update_layout(**BASE_LAYOUT,
                  title="Evidence Evolution: P(Mens Email > Control) vs Sample Size",
                  xaxis=dict(title="Cumulative Sample Size Per Arm", automargin=True),
                  yaxis=dict(title="P(Mens > Control)", automargin=True, tickformat=".0%", range=[0,1.02]),
                  height=480)
fig.write_html(os.path.join(OUT_DIR, "nb06_prob_superiority_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb06_prob_superiority_interactive.html")


  ✓ nb06_prior_posterior_interactive.html


  ✓ nb06_posterior_difference_interactive.html


  ✓ nb06_prob_superiority_interactive.html


### Results-Display Tables (embed-ready go.Table cards for every printed output)

Each card below mirrors one of the printed console blocks and saves as its own
HTML file under `../data/outputs/nb06/` so it can be dropped straight into the
blog post.


In [15]:
# Reusable go.Table card helper (reuses OUT_DIR/PLOTLY_KW/BASE_LAYOUT from Plotly setup cell above)
def table_card(title, header_vals, cell_cols, colwidths,
               row_colors=None, cell_font_size=12, height_extra=80, align="center"):
    n_rows = len(cell_cols[0]) if cell_cols else 0
    stripe = ["#F8F9F9" if i%2==0 else "white" for i in range(n_rows)]
    fill = row_colors if row_colors else [stripe for _ in cell_cols]
    fig = go.Figure(data=[go.Table(
        columnwidth=colwidths,
        header=dict(values=[f"<b>{h}</b>" for h in header_vals],
                    fill_color="#2C3E50",
                    font=dict(color="white", size=13),
                    align="center", height=36),
        cells=dict(values=cell_cols, fill_color=fill, align=align,
                   font=dict(size=cell_font_size, family="monospace"),
                   height=30))])
    fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=20, r=20, t=60, b=20)},
                      title=title,
                      height=36 + 30*n_rows + height_extra)
    return fig

def sig_colors(flags):
    return ["#D5F5E3" if f else "#FADBD8" for f in flags]

def stripe_col(n):
    return ["#F8F9F9" if i%2==0 else "white" for i in range(n)]

# Bayesian A/B testing — Results-Display Tables
from scipy import stats as _sp

RNG = np.random.default_rng(42)
N_SAMPLES = 20000

segments = ["Mens E-Mail", "No E-Mail", "Womens E-Mail"]
# Beta(1,1) uniform prior
alpha0, beta0 = 1, 1
posteriors = {}
for seg in segments:
    g = df_blog[df_blog["segment"]==seg]
    a = alpha0 + g["conversion"].sum()
    b = beta0  + (g["conversion"]==0).sum()
    samples = RNG.beta(a, b, size=N_SAMPLES)
    posteriors[seg] = dict(a=a, b=b, samples=samples, mean=samples.mean(),
                           lo=np.percentile(samples, 2.5),
                           hi=np.percentile(samples, 97.5))

# Card 1: Posterior parameters + summary
rows = []
for seg in segments:
    p = posteriors[seg]
    rows.append((seg, f"{int(p['a'])}", f"{int(p['b'])}", f"{p['mean']:.5f}",
                 f"[{p['lo']:.5f}, {p['hi']:.5f}]", f"{p['hi']-p['lo']:.5f}"))
fig = go.Figure(data=[go.Table(
    columnwidth=[200, 120, 120, 140, 260, 150],
    header=dict(values=[f"<b>{h}</b>" for h in
                        ["Segment","Posterior α","Posterior β","Posterior mean",
                         "95% Credible Interval","CI width"]],
                fill_color="#2C3E50", font=dict(color="white", size=13), align="center", height=36),
    cells=dict(values=list(zip(*rows)), fill_color=[stripe_col(len(rows))]*6,
               align="center", font=dict(size=12, family="monospace"), height=30))])
fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=20, r=20, t=60, b=20)},
                  title="Posterior Summary (Conversion) — Beta-Binomial with Beta(1,1) prior",
                  height=36 + 30*len(rows) + 90)
fig.write_html(os.path.join(OUT_DIR, "nb06_posteriors_interactive.html"), **PLOTLY_KW)
fig.show()

# Card 2: Probability of superiority + expected loss vs Control
control_samples = posteriors["No E-Mail"]["samples"]
rows = []
sig_flags = []
for seg in ["Mens E-Mail", "Womens E-Mail"]:
    s = posteriors[seg]["samples"]
    p_sup = float((s > control_samples).mean())
    lift  = (s - control_samples)
    exp_loss_if_pick = float(np.maximum(0, control_samples - s).mean())
    rows.append((seg,
                 f"{p_sup:.3f}",
                 f"{lift.mean():+.5f}",
                 f"[{np.percentile(lift,2.5):+.5f}, {np.percentile(lift,97.5):+.5f}]",
                 f"{exp_loss_if_pick:.6f}"))
    sig_flags.append(p_sup > 0.95)

fig = go.Figure(data=[go.Table(
    columnwidth=[180, 180, 180, 280, 180],
    header=dict(values=[f"<b>{h}</b>" for h in
                        ["Variant","P(Variant > Control)","Lift (posterior mean)",
                         "95% CrI of lift","Expected loss if chosen"]],
                fill_color="#2C3E50", font=dict(color="white", size=13), align="center", height=36),
    cells=dict(values=list(zip(*rows)),
               fill_color=[sig_colors(sig_flags)] + [stripe_col(len(rows))]*4,
               align="center", font=dict(size=12, family="monospace"), height=30))])
fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=20, r=20, t=60, b=20)},
                  title="Probability of Superiority + Expected Loss (vs No E-Mail)",
                  height=36 + 30*len(rows) + 90)
fig.write_html(os.path.join(OUT_DIR, "nb06_superiority_expected_loss_interactive.html"), **PLOTLY_KW)
fig.show()

# Card 3: Frequentist 95% CI vs Bayesian 95% CrI comparison (difference from control)
from scipy.stats import norm as _norm
rows = []
for seg in ["Mens E-Mail", "Womens E-Mail"]:
    g = df_blog[df_blog["segment"]==seg]
    c = df_blog[df_blog["segment"]=="No E-Mail"]
    p1, n1 = g["conversion"].mean(), len(g)
    p2, n2 = c["conversion"].mean(), len(c)
    se = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
    f_lo, f_hi = (p1-p2)-1.96*se, (p1-p2)+1.96*se
    lift = posteriors[seg]["samples"] - control_samples
    b_lo, b_hi = np.percentile(lift, 2.5), np.percentile(lift, 97.5)
    rows.append((seg, f"{p1-p2:+.5f}",
                 f"[{f_lo:+.5f}, {f_hi:+.5f}]",
                 f"[{b_lo:+.5f}, {b_hi:+.5f}]"))
fig = table_card(
    "Frequentist 95% CI vs Bayesian 95% Credible Interval — Conversion Lift",
    ["Variant vs Control","Point estimate","Frequentist CI (Wald)","Bayesian CrI (95%)"],
    [[r[0] for r in rows], [r[1] for r in rows], [r[2] for r in rows], [r[3] for r in rows]],
    [220, 180, 260, 260])
fig.write_html(os.path.join(OUT_DIR, "nb06_freq_vs_bayes_ci_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb06 Bayesian cards saved")


  ✓ nb06 Bayesian cards saved
